In [4]:
# ----- 0. Importok -----
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from datetime import datetime

In [5]:
# ----- 1. Böngésző indítása -----
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [6]:
# ----- 2. Beolvassuk a linkeket -----
links_df = pd.read_csv("zenga_links.csv")
all_data = []

In [7]:
# ----- 3. Hirdetések feldolgozása (fejlettebb) -----
import re

def extract_number(text):
    """Kiveszi a számot szövegből, pl. '90 millió Ft' -> 90, '113 m²' -> 113"""
    if text is None:
        return None
    text = text.replace("\xa0", " ").replace(".", "")
    match = re.search(r'\d+', text)
    if match:
        return int(match.group())
    return None

for idx, row in links_df.iterrows():
    url = row["url"]
    driver.get(url)
    print(f"\nNyitva: {url}")
    
    time.sleep(3)  # várakozás a betöltődésre
    
    # ---- Title ----
    try:
        title = driver.find_element(By.CSS_SELECTOR, 'h1[data-id="h1"]').text.strip()
        print(f"Title: Találva -> {title}")
    except:
        title = None
        print("Title: Nem található")
    
    # ---- Price ----
    try:
        price_raw = driver.find_element(By.CSS_SELECTOR, '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-main-details > div > div.d-flex.flex-column.flex-gap-sm-16.flex-gap-8 > div.row.ng-star-inserted > div.col-8 > oom-portal-advert-price > div > div > div.fc-black-2.fs-32.fw-900.text-nowrap').text.strip()
        price = extract_number(price_raw)
        print(f"Price: Találva -> {price}")
    except:
        price = None
        print("Price: Nem található")
    
    # ---- Area (m²) ----
    try:
        area_raw = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-first-param"]').text.strip()
        area_m2 = extract_number(area_raw)
        print(f"Area: Találva -> {area_m2}")
    except:
        area_m2 = None
        print("Area: Nem található")
    
    # ---- Floor ----
    try:
        floor = driver.find_element(By.CSS_SELECTOR, '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-main-details > div > div.d-flex.flex-row.justify-content-between.align-items-center.ng-star-inserted > div > div:nth-child(3) > div.text-nowrap.fc-black-2.fs-20.fw-bold.text-center.ng-star-inserted').text.strip()
        print(f"Floor: Találva -> {floor}")
    except:
        floor = None
        print("Floor: Nem található")
    
    # ---- Rooms ----
    try:
        rooms_raw = driver.find_element(By.CSS_SELECTOR, 'div[data-cy="advert-details-second-param"]').text.strip()
        rooms = extract_number(rooms_raw)
        print(f"Rooms: Találva -> {rooms}")
    except:
        rooms = None
        print("Rooms: Nem található")
    
    # ---- Location ----
    try:
        location = driver.find_element(By.CSS_SELECTOR, 'h3[data-id="h3"]').text.strip()
        print(f"Location: Találva -> {location}")
    except:
        location = None
        print("Location: Nem található")
    
    # ---- Description ----
    try:
        description = driver.find_element(By.CSS_SELECTOR, 'span[data-cy="advert-details-description"]').text.strip()
        print("Description: Találva")
    except:
        description = None
        print("Description: Nem található")
    
    # ---- További jellemzők ----
    selectors = {
        "type": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(1) > div:nth-child(1) > div.col-12.fw-bold',
        "total_floors": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(1) > div:nth-child(2) > div.col-12.fw-bold',
        "condition": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(1) > div:nth-child(3) > div.col-12.fw-bold',
        "heating": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(1) > div:nth-child(4) > div.col-12.fw-bold',
        "construction_year": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(2) > div:nth-child(1) > div.col-12.fw-bold',
        "balcony": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(2) > div:nth-child(2) > div.col-12.fw-bold',
        "energy_class": '#main-content > oom-portal-advert-details > div > div > div.d-flex.flex-column.justify-content-center.flex-gap-sm-24.flex-gap-8.ng-star-inserted > div > div > oom-portal-advert-properties > div > div > div.row.fs-16 > div:nth-child(2) > div:nth-child(3) > div.col-12.fw-bold'
    }
    
    features = {}
    for key, sel in selectors.items():
        try:
            val = driver.find_element(By.CSS_SELECTOR, sel).text.strip()
            if key in ["total_floors", "construction_year", "balcony"]:
                val = extract_number(val)
            features[key] = val
            print(f"{key}: Találva -> {val}")
        except:
            features[key] = None
            print(f"{key}: Nem található")
    
    # ---- Lekérdezés dátuma ----
    scrape_date = datetime.today().strftime('%Y-%m-%d')
    
    # ---- Adat gyűjtése ----
    all_data.append({
        "url": url,
        "title": title,
        "price": price,
        "area_m2": area_m2,
        "floor": floor,
        "rooms": rooms,
        "location": location,
        "description": description,
        **features,
        "scrape_date": scrape_date
    })
    
    time.sleep(2)



Nyitva: https://www.zenga.hu/ingatlan/elado-panellakas-budapest-xv-kerulet/8592730?page=1&pos=1&cr=700
Title: Találva -> Budapest XV. kerület eladó panellakás 4 szobás: 77 millió Ft
Price: Találva -> 77
Area: Találva -> 68
Floor: Találva -> földszint
Rooms: Találva -> 4
Location: Találva -> Eladó lakás, Budapest 15. ker.
Description: Találva
type: Találva -> Panellakás
total_floors: Találva -> None
condition: Találva -> Távfűtés egyedi méréssel
heating: Nem található
construction_year: Találva -> 1970
balcony: Találva -> None
energy_class: Nem található

Nyitva: https://www.zenga.hu/ingatlan/elado-teglalakas-budapest-xiv-kerulet-alsorakos/8628719?page=1&pos=2&cr=650
Title: Találva -> Budapest XIV. kerület eladó téglalakás 3 szobás: 98 millió Ft
Price: Találva -> 98
Area: Találva -> 63
Floor: Találva -> földszint
Rooms: Találva -> 3
Location: Találva -> Eladó kertkapcsolatos új építésű lakás családi házas övezetben hőszivattyús fűtéssel, kocsibeállóval
Description: Találva
type: Találv

KeyboardInterrupt: 

In [8]:
# ----- 4. Eredmények mentése -----
df = pd.DataFrame(all_data)
df.to_csv("zenga_listings_details.csv", index=False)
print(f"\nÖsszesen {len(df)} rekord mentve a CSV-be.")


Összesen 9 rekord mentve a CSV-be.


In [9]:
# ----- 5. Böngésző bezárása -----
driver.quit()